# Extra: A Transformer as a CPU — Designed by Hand vs. Trained

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/A1-Neural-CPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_cpu_program.py', 'q_cpu_noise.py', 'q_cpu_small_model.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

LLMs can write programs, but can a Transformer **run** them? In this unit we define a tiny 8-bit CPU (in the style of the Z80/8080) and build Transformers that "are" that CPU:

* Input: the whole program (ROM) + the CPU's current state (registers, flags, PC, memory)
* Output: the state after executing one instruction

The Transformer has to do the **fetch** (use the PC to find the instruction — exactly the kind of table lookup attention is good at), the **decode** and the **execute** (8-bit arithmetic with carry, logic, memory, jumps).

We do it in two ways:

1. **Part 1 — design the weights by hand.** No training at all: we work out, neuron by neuron, a Transformer whose forward pass *is* the CPU, in the spirit of DeepMind's *Tracr* and *Looped Transformers as Programmable Computers*. It is exact, so it can run programs forever.
2. **Part 2 — train a Transformer on examples** (the control group). It learns from data and gets very close, but "very close" is not good enough for a computer.

At the end we compare them and ask: could this run an NES (Famicom) or an Apple II?

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

## A mini CPU: TinyCPU

* Four 8-bit registers `A B C D`, flags `Z` (result is zero) and `CF` (carry/borrow)
* The program lives in a separate ROM of 16 instructions; `PC` is 0–15
* 8 bytes of data memory `M0`–`M7`
* Instruction set (`x`, `y` are registers, `n` is an 8-bit immediate):

| Instruction | Action | Instruction | Action |
|---|---|---|---|
| `NOP` | nothing | `AND y` / `OR y` / `XOR y` | `A = A op y`, sets Z, CF=0 |
| `MOV x,y` | `x = y` | `LD x,[n]` | `x = M[n]` |
| `MVI x,n` | `x = n` | `ST [n],x` | `M[n] = x` |
| `ADD y` | `A = A + y`, sets Z, CF | `JP n` | jump to n |
| `SUB y` | `A = A - y`, sets Z, CF | `JZ n` / `JNZ n` / `JC n` | conditional jump |
| `INC x` / `DEC x` | `x ± 1`, sets Z | `HALT` | stop (state unchanged) |

The state is 15 numbers: `[PC, A, B, C, D, Z, CF, M0..M7]`; each instruction is 4 numbers: `[op, x, y, n]`.

In [ ]:
OPS = ["NOP", "MOV", "MVI", "ADD", "SUB", "INC", "DEC", "AND", "OR", "XOR",
       "LD", "ST", "JP", "JZ", "JNZ", "JC", "HALT"]
OP = {name: i for i, name in enumerate(OPS)}
REGS = "ABCD"
FIELDS = ["PC", "A", "B", "C", "D", "Z", "CF"] + [f"M{i}" for i in range(8)]
N_OPS, ROM_SIZE, N_FIELDS = len(OPS), 16, len(FIELDS)

### Reference emulator

Written in PyTorch, so it can simulate a whole batch of CPUs at once (in parallel on the GPU). We use it to generate training data and to check answers.

In [ ]:
def cpu_step(prog, s):
    """prog: (N, 16, 4), s: (N, 15)  ->  state after executing one instruction (N, 15)"""
    N = s.shape[0]
    ar = torch.arange(N, device=s.device)
    pc = s[:, 0]
    op, x, y, n = prog[ar, pc].unbind(-1)
    R, Z, CF, M = s[:, 1:5], s[:, 5], s[:, 6], s[:, 7:15]
    A, rx, ry = R[:, 0], R[ar, x], R[ar, y]
    newR, newZ, newCF, newM = R.clone(), Z.clone(), CF.clone(), M.clone()
    new_pc = (pc + 1) % ROM_SIZE

    def write(mask, idx, val, zero_flag=True):
        newR[ar[mask], idx[mask]] = val[mask]
        if zero_flag:
            newZ[mask] = (val[mask] == 0).long()

    is_ = lambda name: op == OP[name]
    write(is_("MOV"), x, ry, zero_flag=False)
    write(is_("MVI"), x, n, zero_flag=False)
    zeros = torch.zeros_like(A)
    for name, t in [("ADD", A + ry), ("SUB", A - ry)]:
        m = is_(name)
        write(m, zeros, t & 255)
        newCF[m] = ((t > 255) | (t < 0)).long()[m]
    write(is_("INC"), x, (rx + 1) & 255)
    write(is_("DEC"), x, (rx - 1) & 255)
    for name, v in [("AND", A & ry), ("OR", A | ry), ("XOR", A ^ ry)]:
        m = is_(name)
        write(m, zeros, v)
        newCF[m] = 0
    write(is_("LD"), x, M[ar, n % 8], zero_flag=False)
    m = is_("ST")
    newM[ar[m], (n % 8)[m]] = rx[m]
    jump = is_("JP") | (is_("JZ") & (Z == 1)) | (is_("JNZ") & (Z == 0)) | (is_("JC") & (CF == 1))
    new_pc = torch.where(jump, n % ROM_SIZE, new_pc)
    new_pc = torch.where(is_("HALT"), pc, new_pc)
    return torch.cat([new_pc[:, None], newR, newZ[:, None], newCF[:, None], newM], dim=1)

### Assembler and display helpers

In [ ]:
def assemble(src):
    """Turn assembly source into a (16, 4) ROM, padding the rest with HALT"""
    rom = []
    for line in src.strip().split("\n"):
        line = line.split(";")[0].strip()
        if not line:
            continue
        name, *args = line.replace(",", " ").split()
        args = [a.strip("[]") for a in args]
        op, x, y, n = OP[name.upper()], 0, 0, 0
        reg = lambda a: REGS.index(a.upper())
        if name.upper() in ("MOV",):
            x, y = reg(args[0]), reg(args[1])
        elif name.upper() in ("MVI", "LD"):
            x, n = reg(args[0]), int(args[1])
        elif name.upper() == "ST":
            n, x = int(args[0]), reg(args[1])
        elif name.upper() in ("ADD", "SUB", "AND", "OR", "XOR"):
            y = reg(args[0])
        elif name.upper() in ("INC", "DEC"):
            x = reg(args[0])
        elif name.upper() in ("JP", "JZ", "JNZ", "JC"):
            n = int(args[0])
        rom.append([op, x, y, n])
    rom += [[OP["HALT"], 0, 0, 0]] * (ROM_SIZE - len(rom))
    return torch.tensor(rom)

def show_state(s):
    s = s.tolist()
    return (f"PC={s[0]:2d} A={s[1]:3d} B={s[2]:3d} C={s[3]:3d} D={s[4]:3d} Z={s[5]} CF={s[6]} "
            f"M={s[7:]}")

# Example: compute 6 x 7 with a loop and store the result in M0
MUL = """
    MVI A,0
    MVI B,7
    MVI C,6
    ADD B      ; 3: A += B
    DEC C
    JNZ 3
    ST [0],A
    HALT
"""
rom = assemble(MUL)
s = torch.zeros(1, N_FIELDS, dtype=torch.long)
for t in range(25):
    print(f"{t:2d} {OPS[rom[s[0,0], 0]]:5s}", show_state(s[0]))
    s = cpu_step(rom[None], s)

### Random test cases

To test a CPU implementation we need lots of cases. The next state depends only on the current state and the program, so we can generate an unlimited number of random (program, state) pairs and compute the correct answers with the reference emulator.

In [ ]:
def random_batch(N, device=device):
    op = torch.randint(0, N_OPS, (N, ROM_SIZE), device=device)
    x = torch.randint(0, 4, (N, ROM_SIZE), device=device)
    y = torch.randint(0, 4, (N, ROM_SIZE), device=device)
    n = torch.randint(0, 256, (N, ROM_SIZE), device=device)
    # Keep most jump targets and memory addresses in a sensible range
    small = torch.randint(0, 16, (N, ROM_SIZE), device=device)
    n = torch.where(torch.rand(N, ROM_SIZE, device=device) < 0.5, small, n)
    prog = torch.stack([op, x, y, n], dim=-1)
    s = torch.cat([torch.randint(0, ROM_SIZE, (N, 1), device=device),
                   torch.randint(0, 256, (N, 4), device=device),
                   torch.randint(0, 2, (N, 2), device=device),
                   torch.randint(0, 256, (N, 8), device=device)], dim=1)
    # Set some registers to small numbers, 0 or 255 to get more carry and zero cases
    special = torch.tensor([0, 1, 2, 254, 255], device=device)
    mask = torch.rand(N, 4, device=device) < 0.2
    s[:, 1:5] = torch.where(mask, special[torch.randint(0, 5, (N, 4), device=device)], s[:, 1:5])
    return prog, s, cpu_step(prog, s)

prog, s, s_next = random_batch(3)
print(prog[0, :4])
print(s[0], "\n", s_next[0])

# Part 1: Designing a Transformer CPU by hand

## The idea

A Transformer is a stack of blocks that read from and add to a **residual stream**: a vector per token.
Instead of letting training decide what each dimension means, *we* decide:

* Every quantity gets its own named dimension (`"PC"`, `"A"`, `"M3"`, `"carry"`, `"op_ADD"`, ...), stored as a plain number (an integer, or 0/1 for booleans).
* There are 17 tokens: 16 **ROM tokens** (one per instruction: its address as a one-hot, the opcode as a one-hot, `x`, `y` as one-hots and `n` as a number) and 1 **state token** holding the CPU state.
* Attention moves information between tokens; the MLPs (Linear → ReLU → Linear) compute.
* There is no LayerNorm (Tracr makes the same simplification), and all weights are set by hand.

### Three building blocks made of ReLUs

Everything the CPU needs is built from three tricks. For integers $u$ and booleans $a, b, c \in \{0, 1\}$:

* **AND**: $\mathrm{ReLU}(a + b - 1) = a \wedge b$
* **Gate** (select a value $0 \le v \le 255$ when $c = 1$): $\mathrm{ReLU}(v - 255(1 - c)) = c \cdot v$
* **Lookup table**: $\mathrm{ReLU}(u-k+1) - 2\,\mathrm{ReLU}(u-k) + \mathrm{ReLU}(u-k-1)$ is exactly $1$ when $u = k$ and $0$ for every other integer.
  So a single layer of ReLUs can compute **any function of an integer**, $f(u) = \sum_k f(k)\,[u = k]$ — mod 256, carry, the bits of a byte, "is it zero?", ...

Each ReLU neuron is written as `(input weights, bias, output weights)`, with the weights given as `{dimension name: weight}`:

In [ ]:
from collections import defaultdict

#      weights are dicts {residual-dimension name: weight}
def plus(*ds):
    out = defaultdict(float)
    for d in ds:
        for k, v in d.items():
            out[k] += v
    return dict(out)

def lut(layer, u, lo, hi, outputs):
    """Any function of an integer u in [lo, hi], computed exactly by one ReLU layer.
    u: dict (u is a linear combination of residual dims); outputs: {out_name: f}
    Uses onehot_k(u) = ReLU(u-k+1) - 2 ReLU(u-k) + ReLU(u-k-1)."""
    coef = defaultdict(lambda: defaultdict(float))
    for name, f in outputs.items():
        for k in range(lo, hi + 1):
            v = float(f(k))
            if v:
                coef[k - 1][name] += v
                coef[k][name] -= 2 * v
                coef[k + 1][name] += v
    for j in range(lo - 1, hi + 2):
        out = {n: c for n, c in coef[j].items() if c != 0}
        if out:
            layer.append((u, -j, out))          # neuron ReLU(u - j)

def gate(layer, b, v, out, K=255):
    """out += v if b == 1 else 0      (b in {0,1}, 0 <= v <= K):  ReLU(v - K(1-b))"""
    layer.append((plus(v, {k: K * w for k, w in b.items()}), -K, out))

def and_(layer, a, b, out):
    """out += a AND b   (a, b in {0,1}):  ReLU(a + b - 1)"""
    layer.append((plus(a, b), -1, out))

def copy(layer, x, out):
    """out += x   (x >= 0):  ReLU(x)"""
    layer.append((x, 0, out))

def const(layer, out):
    """out += 1:  ReLU(1)"""
    layer.append(({}, 1, out))

op = {name: {f"op_{name}": 1} for name in OPS}        # fetched opcode, one-hot
X_OPS = ["MOV", "MVI", "INC", "DEC", "LD"]            # write register x
A_OPS = ["ADD", "SUB", "AND", "OR", "XOR"]            # write register A
is_x = plus(*[op[o] for o in X_OPS])
is_a = plus(*[op[o] for o in A_OPS])
bit = lambda i: (lambda t: (t >> i) & 1)

Let's check the lookup-table trick on its own: one MLP layer that computes $u \bmod 4$ and "is $u$ a multiple of 5?" for $u = 0..19$.

In [ ]:
demo = []
lut(demo, {"u": 1}, 0, 19, {"mod4": lambda t: t % 4, "div5": lambda t: t % 5 == 0})
print(len(demo), "ReLU neurons")
u = torch.arange(20).float()
mod4 = sum(F.relu(u * inp.get("u", 0) + b) * out.get("mod4", 0) for inp, b, out in demo)
div5 = sum(F.relu(u * inp.get("u", 0) + b) * out.get("div5", 0) for inp, b, out in demo)
print("u     ", u.long().tolist())
print("u % 4 ", mod4.long().tolist())
print("5 | u ", div5.long().tolist())

## The program: six Transformer blocks

| Block | Attention | MLP |
|---|---|---|
| 1 | — | `PC` → one-hot (the attention query) and `(PC+1) % 16` |
| 2 | **fetch**: the state token attends to the ROM token whose address equals `PC` and copies its `op, x, y, n` | read `R[x]`, `R[y]`; which register to write; jump condition; `n % 16`; one-hot of `n % 8` |
| 3 | — | the ALU: `A+y`, `A-y`, `x±1` with carry/borrow and zero flags (lookup tables on `A+ry`, `A-ry`, `rx`); bits of `A` and `y`; memory load; store enable |
| 4 | — | bitwise `AND / OR / XOR` from the bits |
| 5 | — | select the result of the current instruction |
| 6 | — | write back: add (new − old) to `PC`, registers, flags and memory through the residual connection |

The fetch attention uses query = one-hot(PC), key = one-hot(address) and a huge scale (1000), so the softmax puts weight $1$ on the matching instruction and $e^{-1000} = 0$ on everything else: an exact table lookup.

In [ ]:
blocks = []

# Block 1: no attention. MLP: PC -> one-hot (used as the attention query), (PC+1) % 16
mlp1 = []
lut(mlp1, {"PC": 1}, 0, 15, {**{f"pc_oh{k}": (lambda t, k=k: t == k) for k in range(16)},
                              "pc_next": lambda t: (t + 1) % 16})
blocks.append((None, mlp1))

# Block 2: attention = FETCH. The state token looks for the ROM token whose address == PC
#          and copies its instruction (op, x, y, n). MLP: read registers, decode n, jump condition
fetch = dict(query=[f"pc_oh{k}" for k in range(16)], key=[f"addr{k}" for k in range(16)],
             value=[f"I_op{i}" for i in range(N_OPS)] + [f"I_x{i}" for i in range(4)] +
                   [f"I_y{i}" for i in range(4)] + ["I_n"],
             out=[f"op_{name}" for name in OPS] + [f"x{i}" for i in range(4)] +
                 [f"y{i}" for i in range(4)] + ["n"])
mlp2 = []
for r in range(4):
    gate(mlp2, {f"x{r}": 1}, {REGS[r]: 1}, {"rx": 1})     # rx = R[x]
    gate(mlp2, {f"y{r}": 1}, {REGS[r]: 1}, {"ry": 1})     # ry = R[y]
    and_(mlp2, {f"x{r}": 1}, is_x, {f"dest{r}": 1})       # which register is written
copy(mlp2, is_a, {"dest0": 1})
copy(mlp2, op["JP"], {"jump": 1})                          # jump condition
and_(mlp2, op["JZ"], {"Z": 1}, {"jump": 1})
mlp2.append((plus(op["JNZ"], {"Z": -1}), 0, {"jump": 1}))  # JNZ and not Z
and_(mlp2, op["JC"], {"CF": 1}, {"jump": 1})
lut(mlp2, {"n": 1}, 0, 255, {"n16": lambda t: t % 16,
                             **{f"addr_oh{j}": (lambda t, j=j: t % 8 == j) for j in range(8)}})
blocks.append((fetch, mlp2))

# Block 3: the ALU, all operations computed in parallel
mlp3 = []
lut(mlp3, {"A": 1, "ry": 1}, 0, 510, {"add": lambda t: t % 256, "carry": lambda t: t > 255,
                                       "z_add": lambda t: t % 256 == 0})
lut(mlp3, {"A": 1, "ry": -1}, -255, 255, {"sub": lambda t: t % 256, "borrow": lambda t: t < 0,
                                           "z_sub": lambda t: t % 256 == 0})
lut(mlp3, {"rx": 1}, 0, 255, {"inc": lambda t: (t + 1) % 256, "z_inc": lambda t: (t + 1) % 256 == 0,
                              "dec": lambda t: (t - 1) % 256, "z_dec": lambda t: (t - 1) % 256 == 0})
lut(mlp3, {"A": 1}, 0, 255, {f"a{i}": bit(i) for i in range(8)})       # bits of A
lut(mlp3, {"ry": 1}, 0, 255, {f"b{i}": bit(i) for i in range(8)})      # bits of ry
for j in range(8):
    gate(mlp3, {f"addr_oh{j}": 1}, {f"M{j}": 1}, {"load": 1})          # load = M[n % 8]
    and_(mlp3, op["ST"], {f"addr_oh{j}": 1}, {f"st{j}": 1})            # store into M[j]?
blocks.append((None, mlp3))

# Block 4: bitwise logic
mlp4 = []
for i in range(8):
    w = 2 ** i
    and_(mlp4, {f"a{i}": 1}, {f"b{i}": 1}, {f"and{i}": 1, "and": w, "or": -w, "xor": -2 * w})
    copy(mlp4, {f"a{i}": 1}, {"or": w, "xor": w})
    copy(mlp4, {f"b{i}": 1}, {"or": w, "xor": w})
s_ab = plus(*[{f"a{i}": 1, f"b{i}": 1} for i in range(8)])
const(mlp4, {"z_or": 1})                                   # z_or = 1 - [sum of bits >= 1]
copy(mlp4, s_ab, {"z_or": -1})
mlp4.append((s_ab, -1, {"z_or": 1}))
blocks.append((None, mlp4))

# Block 5: pick the result of the current instruction
mlp5 = []
s_and = plus(*[{f"and{i}": 1} for i in range(8)])
const(mlp5, {"z_and": 1})
copy(mlp5, s_and, {"z_and": -1})
mlp5.append((s_and, -1, {"z_and": 1}))
for name, value in [("MOV", "ry"), ("MVI", "n"), ("INC", "inc"), ("DEC", "dec"), ("LD", "load"),
                    ("ADD", "add"), ("SUB", "sub"), ("AND", "and"), ("OR", "or"), ("XOR", "xor")]:
    gate(mlp5, op[name], {value: 1}, {"result": 1})
blocks.append((None, mlp5))

# Block 6: write back. Outputs are *changes* added to PC, A..D, Z, CF, M0..M7 (residual connection)
mlp6 = []
for r in range(4):
    gate(mlp6, {f"dest{r}": 1}, {"result": 1}, {REGS[r]: 1})      # R[r] += result - R[r]
    gate(mlp6, {f"dest{r}": 1}, {REGS[r]: 1}, {REGS[r]: -1})
for name, z in [("ADD", "z_add"), ("SUB", "z_sub"), ("INC", "z_inc"), ("DEC", "z_dec"),
                ("AND", "z_and"), ("OR", "z_or"), ("XOR", "z_sub")]:   # A xor y == 0  <=>  A == y
    and_(mlp6, op[name], {z: 1}, {"Z": 1})
and_(mlp6, plus(*[op[o] for o in ["ADD", "SUB", "INC", "DEC", "AND", "OR", "XOR"]]), {"Z": 1}, {"Z": -1})
and_(mlp6, op["ADD"], {"carry": 1}, {"CF": 1})
and_(mlp6, op["SUB"], {"borrow": 1}, {"CF": 1})
and_(mlp6, is_a, {"CF": 1}, {"CF": -1})
for j in range(8):
    gate(mlp6, {f"st{j}": 1}, {"rx": 1}, {f"M{j}": 1})
    gate(mlp6, {f"st{j}": 1}, {f"M{j}": 1}, {f"M{j}": -1})
gate(mlp6, {"jump": 1}, {"n16": 1}, {"PC": 1})                     # new PC
gate(mlp6, op["HALT"], {"PC": 1}, {"PC": 1})
mlp6.append((plus({"pc_next": 1}, {"jump": -255}, {k: -255 for k in op["HALT"]}), 0, {"PC": 1}))
copy(mlp6, {"PC": 1}, {"PC": -1})
blocks.append((None, mlp6))

## Compiling into weight matrices

Give each name a dimension of the residual stream, then write every neuron into an ordinary `nn.Linear`. The model below is a normal Transformer forward pass: embeddings, `scaled_dot_product_attention`, Linear-ReLU-Linear MLPs and residual connections.

In [ ]:
names = {}
def dim(name):
    if name not in names:
        names[name] = len(names)
    return names[name]

INPUT_DIMS = FIELDS + [f"addr{k}" for k in range(16)] + [f"I_op{i}" for i in range(N_OPS)] + \
    [f"I_x{i}" for i in range(4)] + [f"I_y{i}" for i in range(4)] + ["I_n"]
for name in INPUT_DIMS:
    dim(name)
for attn, mlp in blocks:
    if attn:
        for k in ("query", "key", "value", "out"):
            for name in attn[k]:
                dim(name)
    for inp, b, out in mlp:
        for name in list(inp) + list(out):
            dim(name)
D_MODEL = len(names)


class DesignedBlock(nn.Module):
    """A standard pre-norm-free Transformer block: x = x + Attn(x);  x = x + MLP(x)"""
    def __init__(self, attn, mlp, beta=1000.0):
        super().__init__()
        self.beta = beta
        if attn:
            dq = len(attn["query"])
            self.Wq = nn.Linear(D_MODEL, dq, bias=False)
            self.Wk = nn.Linear(D_MODEL, dq, bias=False)
            self.Wv = nn.Linear(D_MODEL, len(attn["value"]), bias=False)
            self.Wo = nn.Linear(len(attn["value"]), D_MODEL, bias=False)
            for W in (self.Wq, self.Wk, self.Wv, self.Wo):
                nn.init.zeros_(W.weight)
            for i, (q, k) in enumerate(zip(attn["query"], attn["key"])):
                self.Wq.weight.data[i, names[q]] = 1.0
                self.Wk.weight.data[i, names[k]] = 1.0
            for i, (v, o) in enumerate(zip(attn["value"], attn["out"])):
                self.Wv.weight.data[i, names[v]] = 1.0
                self.Wo.weight.data[names[o], i] = 1.0
        else:
            self.Wq = None
        self.fc1 = nn.Linear(D_MODEL, len(mlp))
        self.fc2 = nn.Linear(len(mlp), D_MODEL, bias=False)
        nn.init.zeros_(self.fc1.weight); nn.init.zeros_(self.fc2.weight)
        for h, (inp, b, out) in enumerate(mlp):
            for name, w in inp.items():
                self.fc1.weight.data[h, names[name]] = w
            self.fc1.bias.data[h] = b
            for name, w in out.items():
                self.fc2.weight.data[names[name], h] = w

    def forward(self, x):
        if self.Wq is not None:
            q, k, v = self.Wq(x), self.Wk(x), self.Wv(x)
            # very large scale => softmax becomes an exact "argmax" (hard attention)
            x = x + self.Wo(F.scaled_dot_product_attention(q, k, v, scale=self.beta))
        return x + self.fc2(F.relu(self.fc1(x)))


class DesignedCPU(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList([DesignedBlock(a, m) for a, m in blocks])

    def embed(self, prog, s):
        """tokens 0..15 = ROM, token 16 = CPU state; numbers are stored as plain integers"""
        N = s.shape[0]
        x = torch.zeros(N, ROM_SIZE + 1, D_MODEL, device=s.device)
        op_, rx_, ry_, n_ = prog.unbind(-1)
        r = torch.arange(ROM_SIZE, device=s.device)
        x[:, r, names["addr0"] + r] = 1.0
        ones = torch.ones(N, ROM_SIZE, 1, device=s.device)
        x[:, :ROM_SIZE].scatter_add_(2, (names["I_op0"] + op_)[..., None], ones)   # one-hot opcode
        x[:, :ROM_SIZE].scatter_add_(2, (names["I_x0"] + rx_)[..., None], ones)    # one-hot x
        x[:, :ROM_SIZE].scatter_add_(2, (names["I_y0"] + ry_)[..., None], ones)    # one-hot y
        x[:, :ROM_SIZE, names["I_n"]] = n_.float()
        x[:, ROM_SIZE, :N_FIELDS] = s.float()               # FIELDS are dims 0..14
        return x

    def forward(self, prog, s):
        x = self.embed(prog, s)
        for blk in self.blocks:
            x = blk(x)
        return x[:, ROM_SIZE, :N_FIELDS]

    @torch.no_grad()
    def step(self, prog, s):
        return self(prog, s).round().long()

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = False   # keep full fp32 precision (TF32 would round the integers)
designed = DesignedCPU().to(device)
n_params = sum(p.numel() for p in designed.parameters())
n_nonzero = sum((p != 0).sum().item() for p in designed.parameters())
print(f"d_model = {D_MODEL}, MLP widths = {[len(m) for _, m in blocks]}")
print(f"parameters: {n_params:,} (of which non-zero: {n_nonzero:,})")

### Looking inside: the fetch attention

Run the 6 x 7 program up to `PC = 3` and look at where the state token attends:

In [ ]:
rom = assemble(MUL)[None].to(device)
s = torch.tensor([[3, 0, 7, 6, 0, 0, 0] + [0] * 8], device=device)    # PC = 3
x = designed.embed(rom, s)
x = designed.blocks[0](x)                     # block 1 computes the one-hot of PC
blk = designed.blocks[1]
scores = blk.beta * blk.Wq(x)[0, -1] @ blk.Wk(x)[0].T
print("attention weights of the state token over the 17 tokens:")
print(scores.softmax(-1).detach().cpu().numpy().round(3))

## Is it exact?

Compare with the reference emulator on 200,000 random programs and states, covering every instruction:

In [ ]:
@torch.no_grad()
def exact_rate(stepper, batch_fn, N=200_000, chunk=20_000):
    ok, per_op = [], torch.zeros(N_OPS, 2)
    for i in range(0, N, chunk):
        prog, s, target = batch_fn(chunk)
        good = (stepper(prog, s) == target).all(1)
        op = prog[torch.arange(chunk, device=device), s[:, 0], 0]
        per_op[:, 0] += torch.bincount(op[good], minlength=N_OPS).cpu()
        per_op[:, 1] += torch.bincount(op, minlength=N_OPS).cpu()
        ok.append(good)
    return torch.cat(ok).float().mean().item(), {OPS[i]: (per_op[i, 0] / per_op[i, 1]).item() for i in range(N_OPS)}

acc_designed, per_op_designed = exact_rate(designed.step, random_batch)
print(f"designed CPU, single-step exact accuracy: {acc_designed:.6f}")
print("worst instruction:", min(per_op_designed.items(), key=lambda kv: kv[1]))

## Running real programs

`run` executes a CPU step by step, feeding each output back as the next input; `compare` runs a program on the reference emulator and on every CPU in `CPUS`, and reports where (if anywhere) they diverge.

In [ ]:
def run(prog, s, steps, stepper):
    traj = [s]
    for _ in range(steps):
        s = stepper(prog, s)
        traj.append(s)
    return torch.stack(traj, dim=1)   # (N, steps+1, 15)

CPUS = {"designed": designed.step}

def compare(src, steps=100, name=""):
    rom = assemble(src)[None].to(device)
    s0 = torch.zeros(1, N_FIELDS, dtype=torch.long, device=device)
    ref = run(rom, s0, steps, cpu_step)[0]
    print(f"{name}  reference      {show_state(ref[-1])}")
    for cpu_name, stepper in CPUS.items():
        out = run(rom, s0, steps, stepper)[0]
        diff = (ref != out).any(1).nonzero()
        status = "all steps match" if len(diff) == 0 else f"diverges at step {diff[0].item()}"
        print(f"{' ' * len(name)}  {cpu_name:13s}  {show_state(out[-1])}   {'✅' if len(diff) == 0 else '❌'} {status}")

FIB = """
    MVI A,1
    MVI B,0
    MVI D,10
    MOV C,A     ; 3
    ADD B
    MOV B,C
    DEC D
    JNZ 3
    ST [1],A
    HALT
"""
SUM = """
    MVI A,0
    MVI B,20
    ADD B       ; 2: A += B
    DEC B
    JNZ 2
    ST [2],A
    HALT
"""
compare(MUL, 40, "6x7")
compare(FIB, 60, "fib")
compare(SUM, 80, "1+..+20")

Being exact means it can run as long as we like. Run 1,000 random programs for 10,000 steps each and compare every step:

In [ ]:
torch.manual_seed(1)
prog, s, _ = random_batch(1000)
s_ref, s_designed = s.clone(), s.clone()
diverged = torch.zeros(len(s), dtype=torch.bool, device=device)
t0 = time.perf_counter()
for t in range(10_000):
    s_ref, s_designed = cpu_step(prog, s_ref), designed.step(prog, s_designed)   # both run on their own
    diverged |= (s_designed != s_ref).any(1)
print(f"10,000 steps x 1,000 programs: {diverged.sum().item()} programs diverged  ({time.perf_counter()-t0:.0f}s)")

## Q
How fragile is a hand-designed circuit? Add Gaussian noise with standard deviation $\sigma$ to every weight of the designed CPU (on a copy) and measure the single-step exact accuracy for $\sigma = 10^{-6}, 10^{-4}, 10^{-3}, 10^{-2}, 10^{-1}$.
Which parts of the design break first? (Hint: think about the factor 255 in the gate and the huge attention scale.)

In [ ]:
# %load q_cpu_noise.py
%run -i q_cpu_noise.py

# Part 2 (control group): training a Transformer CPU

Now forget the design and let a standard Transformer **learn** the same CPU from examples.

Completely random states look very different from the states a real program runs into: real programs usually start from an all-zero state, are padded with `HALT`, and use smaller numbers.
So besides `random_batch` we also generate "realistic program" data: random program length with the rest padded with `HALT`, start from the all-zero state, run a random number of steps, then use the next step as the question. Training uses half of each kind.

In [ ]:
HALT_INS = torch.tensor([OP["HALT"], 0, 0, 0], device=device)

def program_batch(N, device=device, max_warmup=24):
    prog, _, _ = random_batch(N, device)
    length = torch.randint(3, ROM_SIZE + 1, (N, 1), device=device)
    prog[torch.arange(ROM_SIZE, device=device)[None] >= length] = HALT_INS
    s = torch.zeros(N, N_FIELDS, dtype=torch.long, device=device)
    for _ in range(np.random.randint(0, max_warmup)):
        s = cpu_step(prog, s)
    return prog, s, cpu_step(prog, s)

def mixed_batch(N):
    a, b = random_batch(N // 2), program_batch(N - N // 2)
    return tuple(torch.cat([x, y]) for x, y in zip(a, b))

## Model: a Transformer CPU

The input is 16 "instruction tokens" + 15 "state tokens", 31 tokens in total:

* **Numbers are represented in binary**: a byte is split into 8 bits and mapped to a vector by a `Linear`; the output also predicts 8 bits. This makes the carry rule of addition much easier to learn (far easier than treating 0–255 as 256 unrelated symbols)
* Instruction token = opcode embedding + register embeddings + immediate-value bits + **address embedding** (so attention can use the PC to find the instruction)
* State token = value bits + field embedding (is this the PC, A, or M3, ...?)
* A bidirectional (non-causal) Transformer encoder: the next values of all 15 state fields are predicted **at the same time**

In [ ]:
def to_bits(v):
    return ((v[..., None] >> torch.arange(8, device=v.device)) & 1).float()

def from_bits(logits):
    return ((logits > 0).long() << torch.arange(8, device=logits.device)).sum(-1)

class NeuralCPU(nn.Module):
    def __init__(self, d=256, n_layers=6, n_heads=8):
        super().__init__()
        self.op_emb = nn.Embedding(N_OPS, d)
        self.x_emb = nn.Embedding(4, d)
        self.y_emb = nn.Embedding(4, d)
        self.addr_emb = nn.Embedding(ROM_SIZE, d)
        self.field_emb = nn.Embedding(N_FIELDS, d)
        self.bits_in = nn.Linear(8, d)
        layer = nn.TransformerEncoderLayer(d, n_heads, 4 * d, dropout=0.0, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.norm = nn.LayerNorm(d)
        self.bits_out = nn.Linear(d, 8)

    def forward(self, prog, s):
        op, x, y, n = prog.unbind(-1)
        addr = torch.arange(ROM_SIZE, device=prog.device)
        fields = torch.arange(N_FIELDS, device=prog.device)
        prog_tok = self.op_emb(op) + self.x_emb(x) + self.y_emb(y) + self.bits_in(to_bits(n)) + self.addr_emb(addr)
        state_tok = self.bits_in(to_bits(s)) + self.field_emb(fields)
        h = self.encoder(torch.cat([prog_tok, state_tok], dim=1))
        return self.bits_out(self.norm(h[:, ROM_SIZE:]))       # (N, 15, 8) bit logits

    @torch.no_grad()
    def step(self, prog, s):
        with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
            return from_bits(self(prog, s).float())

model = NeuralCPU().to(device)
print(f"number of parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## Training data: random programs + random states

The next state depends only on the current state and the program, so we can generate an unlimited number of random (program, state) pairs and compute the answers directly on the GPU with the reference emulator.

In [ ]:
@torch.no_grad()
def evaluate(model, N=20000, batch_fn=None, chunk=4096):
    model.eval()
    prog, s, target = (batch_fn or mixed_batch)(N)
    pred = torch.cat([model.step(prog[i:i + chunk], s[i:i + chunk]) for i in range(0, N, chunk)])
    ok = (pred == target).all(1)
    op = prog[torch.arange(N, device=device), s[:, 0], 0]
    per_op = {OPS[i]: ok[op == i].float().mean().item() for i in range(N_OPS)}
    return ok.float().mean().item(), per_op

def train(model, steps, batch_size=512, lr=1e-3, log_every=1000):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=steps, pct_start=0.05)
    scaler = torch.amp.GradScaler(device, enabled=(device == "cuda"))
    t0 = time.perf_counter()
    for step in range(1, steps + 1):
        model.train()
        prog, s, target = mixed_batch(batch_size)
        with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(prog, s)
        loss = F.binary_cross_entropy_with_logits(logits.float(), to_bits(target))
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        if step % log_every == 0 or step == steps:
            acc, _ = evaluate(model, 4096)
            print(f"step {step:5d}  loss {loss.item():.5f}  single-step exact accuracy {acc:.4f}  {time.perf_counter()-t0:.0f}s")

TRAIN_STEPS = 15000
train(model, TRAIN_STEPS)
CPUS["trained"] = model.step

## Comparison 1: single-step accuracy

In [ ]:
acc_trained, per_op_trained = exact_rate(model.step, random_batch)
print(f"single-step exact accuracy   designed: {acc_designed:.6f}   trained: {acc_trained:.6f}")
print(f"  trained, realistic programs: {exact_rate(model.step, program_batch, N=40_000)[0]:.6f}")
for name in sorted(per_op_trained, key=per_op_trained.get):
    print(f"  {name:5s} designed {per_op_designed[name]:.5f}   trained {per_op_trained[name]:.5f}")

## Comparison 2: running programs

In [ ]:
compare(MUL, 40, "6x7")
compare(FIB, 60, "fib")
compare(SUM, 80, "1+..+20")

## Comparison 3: errors accumulate

Even with 99.9% single-step accuracy, the probability that all $k$ steps are correct is about $0.999^k$. A real computer executes millions of instructions per second, and **a single wrong bit can crash the program**.

Run 2000 randomly generated programs (starting from the all-zero state) for 100 steps each, and look at the fraction that is "still exactly correct up to step k":

In [ ]:
N, K = 2000, 100
prog, s0, _ = program_batch(N, max_warmup=1)
ref = run(prog, s0, K, cpu_step)
for cpu_name, stepper in CPUS.items():
    traj = torch.cat([run(prog[i:i + 500], s0[i:i + 500], K, stepper) for i in range(0, N, 500)])
    ok_until = (ref == traj).all(-1).cumprod(1).float().mean(0).cpu()
    plt.plot(ok_until, label=cpu_name)
    print(f"{cpu_name:9s}: fraction still exactly correct after {K} steps = {ok_until[-1]:.3f}")
p1 = acc_trained
plt.plot([p1 ** k for k in range(K + 1)], "--", label=f"{p1:.4f}^k")
plt.xlabel("steps"); plt.ylabel("fraction still exactly correct"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Q
Write your own program (e.g. compute $2^7$, take the maximum of two numbers, or compute the quotient of 100 divided by 7), and run it on the reference emulator, the designed CPU and the trained CPU.

In [ ]:
# %load q_cpu_program.py
%run -i q_cpu_program.py

## Q
If the model has only **2 layers**, which instructions are hardest to learn? Why? (Hint: the carry in 8-bit addition may have to propagate all the way from bit 0 to bit 7.)

In [ ]:
# %load q_cpu_small_model.py
%run -i q_cpu_small_model.py

## Comparison 4: speed — could it run an NES?

In [ ]:
def speed(stepper, B):
    prog, s, _ = random_batch(B)
    stepper(prog, s)   # warm up
    if device == "cuda": torch.cuda.synchronize()
    t0, n = time.perf_counter(), 50 if B == 1 else 10
    for _ in range(n):
        s = stepper(prog, s)
    if device == "cuda": torch.cuda.synchronize()
    return n * B / (time.perf_counter() - t0)

nes = 1.79e6 / 3.5    # the NES's 6502: 1.79 MHz, about 3.5 cycles per instruction on average
print(f"NES 6502: {nes:12,.0f} instructions/s;  one frame (1/60 s) is about {nes/60:,.0f} instructions")
for cpu_name, stepper in CPUS.items():
    single, batched = speed(stepper, 1), speed(stepper, 8192)
    print(f"{cpu_name:9s} one CPU: {single:9,.0f} instr/s ({nes / single:,.0f}x slower than an NES, "
          f"{nes / 60 / single:,.0f} s per NES frame);  8192 CPUs in parallel: {batched:12,.0f} instr/s in total")

## Conclusion: designed vs. trained, and could a Transformer run a Z80, an NES or an Apple II?

| | Designed by hand (Part 1) | Trained (Part 2) |
|---|---|---|
| How it was made | a person worked out every neuron | gradient descent on random examples |
| Single-step accuracy | exact (100%) | very high but not 100% |
| Long programs | runs forever without a single error | errors accumulate ($p^k$) |
| Parameters | ~320K, of which only ~4.7K are non-zero | ~4.8M, all used |
| Robustness to weight noise | fragile (see the Q) | smoother |
| Effort to extend (new instruction, bigger CPU) | design new circuits by hand | generate data and train again |

Scaling up:

| | TinyCPU (this unit) | Z80 / 6502 | NES / Apple II |
|---|---|---|---|
| Registers | four 8-bit | 7–14, including 16-bit ones | 6502: A, X, Y, SP, P, PC |
| Memory | 8 bytes + 16 instructions | 64KB address space | NES: 2KB RAM + 32KB+ cartridge + PPU graphics chip; Apple II: 48KB RAM with memory-mapped display |
| Instructions | 17 | Z80 about 700 (with prefixes), 6502 151 | also need to emulate the PPU, sound, timing, mappers |

* **Designed**: nothing in principle stops a hand-designed Transformer from executing a Z80 or 6502 exactly. Memory would become tokens read with attention (like our fetch), 16-bit arithmetic would be built from bits and carries over a few more layers, and every instruction would get its own circuit. The real obstacle for an NES or Apple II is **speed**: one forward pass per instruction is thousands of times too slow (see Comparison 4), before even emulating the graphics chip.
* **Trained**: a Transformer can learn single steps very well, but **exactness over long runs** is the hard part. This is one reason LLMs make mistakes in long chains of exact computation, and why they usually **call tools** for exact work (see tool calling in 04-LLM-Apps).
* Related work: *Tracr* (DeepMind, 2023) compiles RASP programs into Transformer weights; *Looped Transformers as Programmable Computers* (2023) runs a SUBLEQ computer by looping a hand-designed Transformer, just like our step-by-step loop.
* If the goal is something that "looks like playing the game", see **world models**: GameNGen (2024) generates DOOM frames with a diffusion model and Oasis (2024) generates Minecraft. They imitate the screen rather than executing the game code, and training them takes far more compute than a T4.